# Clasificación de Lesiones Cutáneas — SSL Domain-Adaptive + Fine-tuning (v4 · resume)

**Proyecto de Grado UNAB · Objetivo 2 — v4: reusa el backbone SSL ya entrenado (skip 6h), SSL primero, memoria blindada.**

> **v4 vs v3:** (1) RESUME del backbone SSL desde Input (salta el pretrain de ~6h); (2) brazo **SSL primero** → el modelo entregable se exporta temprano; (3) fix de OOM (`persistent_workers=False`, `gc` entre brazos, `N_SEEDS=2`).

### Qué cambia frente a la v2 (todas las mejoras aplicadas)
1. **Más datos NO etiquetados para el SSL** — pool = **ISIC 2020** (~33k) + HAM(train). El SSL escala con datos.
2. **Validación externa REAL y anti-leakage** — sobre **ISIC 2019** etiquetado, excluyendo **toda imagen que esté en HAM** (HAM ⊂ ISIC2019, mismos IDs `ISIC_*`) y excluyendo esas imágenes del pool SSL. Es la prueba de generalización más fuerte y ahora es honesta.
3. **Ablación multi-semilla (N_SEEDS=3)** — se reporta **media ± std** de macro-F1 y AUC en ambos brazos. Mata la crítica de "Δ=0.0007 con n=1 es ruido".
4. **Robustez one-shot** — backbone SSL con checkpoint + resume, export incremental del modelo entregable, y **todo bloque secundario (probe, Grad-CAM, val externa) es non-fatal**: si uno falla, la corrida NO se cae.
5. **LP-FT + LLRD** (Kumar 2022) preservado — no distorsiona la representación SSL.

### ⚠️ Antes de la corrida real
- **Add Input (Datasets):** `HAM10000` (kmader) · `ISIC 2020 JPG 256×256 RESIZED` (nischaydnk, pool SSL) · `ISIC 2019 JPG 224×224 RESIZED` (nischaydnk, val externa con su CSV de ground truth).
- **Settings → Accelerator: GPU T4 x2.**
- **PRIMERO corré con `SMOKE=True`** (≈3-5 min, valida TODO el pipeline). Si llega al final sin error → poné `SMOKE=False` y **Save & Run All (Commit)** para la corrida real.


In [ ]:
import io, os, gc, json, random, time
from pathlib import Path
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import resnet50, ResNet50_Weights
from PIL import Image, ImageFile
from scipy.special import softmax
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (f1_score, accuracy_score, classification_report,
                             roc_auc_score, roc_curve, confusion_matrix, recall_score)

ImageFile.LOAD_TRUNCATED_IMAGES = True  # una imagen truncada en Kaggle NO debe tumbar el run
T_START = time.time()

def load_rgb(path):
    """Carga robusta: si una imagen esta corrupta, avisa UNA vez y devuelve placeholder negro."""
    try:
        return Image.open(path).convert("RGB")
    except Exception as e:
        if not getattr(load_rgb, "_warned", False):
            print(f"[WARN] imagen ilegible {path}: {e} (se usa placeholder; se silencian las proximas)")
            load_rgb._warned = True
        return Image.new("RGB", (224, 224), (0, 0, 0))

def set_seed(seed=42):
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    np.random.seed(seed); random.seed(seed)
    torch.backends.cudnn.benchmark = True  # velocidad; determinismo pleno no es critico aqui

def elapsed():
    return f"{(time.time()-T_START)/60:.1f} min"

def save_json(obj, path):
    with open(path, "w") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False, default=float)

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
N_GPU = torch.cuda.device_count()
USE_DP = N_GPU > 1  # T4 x2 -> DataParallel para ~2x throughput sin cambiar resultados
print("PyTorch:", torch.__version__, "| Device:", device, "| GPUs:", N_GPU, "| DataParallel:", USE_DP)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

MEAN = [0.485, 0.456, 0.406]; STD = [0.229, 0.224, 0.225]
WORK = Path("/kaggle/working")

# ---- Presupuesto de computo (una sola corrida) ----
# FLUJO: (1) SMOKE=True -> Run All (~3-5 min). Si termina sin error, el pipeline sirve.
#        (2) SMOKE=False -> Save & Run All (Commit) real.
SMOKE       = False  # corrida REAL (ya validado en SMOKE end-to-end). Poner True para re-validar rapido.
RUN_SSL     = True   # False = saltar SSL y correr solo baseline ImageNet
NUM_WORKERS = 4      # Kaggle da ~4 vCPU con GPU

SSL_EPOCHS  = 30     # SimSiam domain-adaptive (desde init ImageNet). 30 da margen de tiempo vs 12h Kaggle
SSL_BATCH   = 128    # batch 128 a 224px entra en T4; SimSiam es robusto a batch chico
SSL_RES     = 224    # MISMA resolucion que el fine-tune -> evita BN drift (arregla el linear probe)
SUP_RES     = 224

N_SEEDS     = 2      # ablacion multi-semilla -> media +/- std. 2 = seguro vs OOM (un 3er seed se agrega luego)
SEEDS       = [42, 1234, 2026][:N_SEEDS]

EXT_VAL_MAXN = 8000  # cap de imagenes de validacion externa (inferencia; None = todas)
MAX_HOURS    = 11.0  # guarda de tiempo: si se pasa, se saltan semillas extra (no silencioso)

if SMOKE:
    SSL_EPOCHS = 2
    N_SEEDS, SEEDS = 1, [42]
    EXT_VAL_MAXN = 400
    print("\n*** MODO SMOKE ACTIVO -> numeros NO validos como resultado, solo valida que el pipeline corre entero. ***")

print(f"Config: SSL_EPOCHS={SSL_EPOCHS} | SEEDS={SEEDS} | EXT_VAL_MAXN={EXT_VAL_MAXN} | workers={NUM_WORKERS}")


## 1. Auto-localizar datasets y separar sus roles (anti-leakage)

Tres roles, **disjuntos**:
- **HAM10000** → etiquetado, split interno train/val.
- **ISIC 2020** (u otro ISIC sin labels 7-clase) → **pool SSL** no etiquetado.
- **ISIC 2019** (el que traiga un CSV de ground truth con columnas `MEL/NV/BCC/AK/BKL/DF/VASC/...`) → **validación externa**. Se excluye del pool SSL y se le quitan las imágenes que están en HAM.

In [ ]:
INPUT = Path("/kaggle/input")

# ---- HAM10000: metadata + imagenes (rol: train/val etiquetado) ----
ham_metas = list(INPUT.glob("**/HAM10000_metadata.csv"))
assert ham_metas, "Falta HAM10000 (kmader). Add Input -> Datasets -> Skin Cancer MNIST: HAM10000"
ham_meta_csv = ham_metas[0]
HAM_DIR = ham_meta_csv.parent
img_paths = {p.stem: p for p in HAM_DIR.glob("**/*.jpg")}
print("HAM metadata:", ham_meta_csv)
print("Imagenes HAM:", len(img_paths))

# ---- Diccionarios de mapeo a las 7 clases HAM (SCC/UNK -> None = descartar) ----
ONEHOT = {"MEL": "mel", "NV": "nv", "BCC": "bcc", "AK": "akiec", "AKIEC": "akiec",
          "BKL": "bkl", "DF": "df", "VASC": "vasc", "SCC": None, "UNK": None}
def name2ham(v):
    """Mapea un valor categorico (codigo corto o nombre completo) a clase HAM o None."""
    s = str(v).strip().lower()
    if s in ("mel", "melanoma"): return "mel"
    if s in ("nv", "nevus", "melanocytic nevus", "nevo"): return "nv"
    if s in ("bcc", "basal cell carcinoma"): return "bcc"
    if s in ("ak", "akiec", "actinic keratosis", "bowen", "intraepithelial carcinoma",
             "actinic keratosis / bowen", "akiec (actinic keratoses)"): return "akiec"
    if s in ("bkl", "benign keratosis", "seborrheic keratosis", "solar lentigo",
             "lichenoid keratosis", "benign keratosis-like lesions"): return "bkl"
    if s in ("df", "dermatofibroma"): return "df"
    if s in ("vasc", "vascular lesion", "vascular", "angioma", "hemangioma"): return "vasc"
    return None  # scc, unknown, etc.

# ---- Detectar el CSV de validacion externa: el que MAS cubra las 7 clases HAM ----
# Robusto a: one-hot (MEL/NV/BCC/...) o columna categorica (diagnosis/dx/label/target).
# Se elige por COBERTURA (>=5 clases) -> descarta el metadata de ISIC2020 (solo mel/nv/unknown).
def parse_gt(csvp):
    try:
        df = pd.read_csv(csvp)
    except Exception:
        return None
    up = {c.upper(): c for c in df.columns}
    onehot = [up[k] for k in up if k in ONEHOT]
    if len(onehot) >= 5:                       # formato one-hot
        idcol = next((c for c in df.columns if c not in onehot), df.columns[0])
        codes = np.array([c.upper() for c in onehot])[df[onehot].values.argmax(1)]
        ham = [ONEHOT.get(c) for c in codes]
    else:                                      # formato columna categorica
        idcol = next((up[k] for k in ("IMAGE", "IMAGE_NAME", "ISIC_ID", "IMAGE_ID", "NAME", "ID")
                      if k in up), df.columns[0])
        labcol = next((c for c in df.columns
                       if c != idcol and df[c].dropna().map(name2ham).notna().mean() > 0.5), None)
        if labcol is None:
            return None
        ham = [name2ham(v) for v in df[labcol]]
    out = pd.DataFrame({"id_stem": df[idcol].astype(str).str.replace(".jpg", "", regex=False),
                        "ham": ham}).dropna(subset=["ham"])
    return out if out["ham"].nunique() >= 5 else None

ext_gt, ext_dir, _best = None, None, -1
print("\n[Val externa] escaneando CSVs candidatos (se excluye el metadata de HAM):")
for csvp in INPUT.glob("**/*.csv"):
    if HAM_DIR in csvp.parents or csvp == ham_meta_csv:
        continue  # el CSV de HAM NO es validacion externa (es el train/val interno)
    try:
        cols = list(pd.read_csv(csvp, nrows=1).columns)
    except Exception as e:
        print(f"  - {csvp.name}: ILEGIBLE ({e})"); continue
    cand = parse_gt(csvp)
    verdict = f"OK n={len(cand)} clases={cand.ham.nunique()}" if cand is not None else "RECHAZADO (no mapea >=5 clases HAM)"
    print(f"  - {csvp.name}: cols={cols[:12]} -> {verdict}")
    if cand is not None and len(cand) > _best:
        ext_gt, ext_dir, _best = cand, csvp.parent, len(cand)
        _best_csv = csvp
if ext_gt is not None:
    print(f"\n[Val externa] ground truth elegido: {_best_csv}")
    print(f"[Val externa] carpeta externa (excluida del SSL): {ext_dir}")
    print("[Val externa] distribucion cruda (mapeada a HAM):", dict(ext_gt.ham.value_counts()))
else:
    print("\n[Val externa] NO se detecto un CSV que cubra >=5 clases HAM.")
    print("  -> quedara OMITIDA (non-fatal). Revisa que el ISIC2019 traiga su metadata con diagnostico.")
    print("  CSVs vistos:", [str(p) for p in INPUT.glob("**/*.csv")][:10])

# ids reservados a la validacion externa -> NUNCA entran al SSL (todos los ids etiquetados)
external_val_ids = set(ext_gt.id_stem) if ext_gt is not None else set()
if external_val_ids:
    print(f"[Val externa] ids etiquetados reservados (excluidos del SSL): {len(external_val_ids)}")

# ---- Pool SSL: TODA ISIC_*.jpg que NO sea HAM ni la carpeta de val externa ----
ssl_extra = {}
for p in INPUT.glob("**/ISIC_*.jpg"):
    if HAM_DIR in p.parents:
        continue
    if ext_dir is not None and ext_dir in p.parents:  # reservado a val externa
        continue
    if p.stem in img_paths or p.stem in external_val_ids:  # dedup por id (HAM ⊂ ISIC2019)
        continue
    ssl_extra[p.stem] = p
print(f"\nImagenes externas NO etiquetadas para el pool SSL (ISIC2020/otras): {len(ssl_extra)}")
if len(ssl_extra) == 0:
    print("  [OJO] pool SSL externo vacio -> el SSL correria solo con HAM train. "
          "Revisa que ISIC2020 tenga imagenes ISIC_*.jpg.")


## 2. Metadata, split lesión-disjunto (anti-leakage) y validación externa mapeada

**Clave anti-leakage:** split de HAM por `lesion_id` (no por `image_id`); y la val externa **excluye todo id presente en HAM** (mismas imágenes con IDs `ISIC_*`).

In [ ]:
meta = pd.read_csv(ham_meta_csv)
meta = meta[meta.image_id.isin(img_paths)].reset_index(drop=True)
classes = sorted(meta.dx.unique())
cls2idx = {c: i for i, c in enumerate(classes)}
id2label = {r.image_id: cls2idx[r.dx] for r in meta.itertuples()}
ham_all_ids = set(meta.image_id)
print("Clases:", classes)
print(meta.dx.value_counts(), "\n")

all_ids = meta.image_id.tolist()
y       = np.array([id2label[i] for i in all_ids])
groups  = meta.set_index("image_id").loc[all_ids, "lesion_id"].values

sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
tr_idx, va_idx = next(sgkf.split(all_ids, y, groups))
tr_ids = [all_ids[i] for i in tr_idx]
va_ids = [all_ids[i] for i in va_idx]

tr_les = set(meta.set_index("image_id").loc[tr_ids, "lesion_id"])
va_les = set(meta.set_index("image_id").loc[va_ids, "lesion_id"])
overlap = tr_les & va_les
print(f"Train imgs: {len(tr_ids)} | Val imgs: {len(va_ids)} | Lesiones compartidas: {len(overlap)}")
assert len(overlap) == 0, "LEAKAGE: hay lesiones en train y val a la vez"

# ---- Validacion externa: construir (path, y) ANTI-LEAKAGE ----
# Regla: label mapeada a las 7 clases HAM + imagen presente en disco + id NO en HAM.
ext_items = []                 # lista de (Path, y_int)
ext_dropped_ham = 0
if ext_gt is not None and ext_dir is not None:
    # imagenes externas: buscar en TODO el input (no solo la carpeta del GT) -> permite que el
    # ground truth venga como CSV suelto y las imagenes vivan en el dataset resized que ya tenes.
    ext_paths = {p.stem: p for p in INPUT.glob("**/ISIC_*.jpg") if HAM_DIR not in p.parents}
    for r in ext_gt.itertuples():
        if r.ham not in cls2idx:               # clase no presente en este HAM (raro)
            continue
        if r.id_stem not in ext_paths:         # sin imagen en disco
            continue
        if r.id_stem in ham_all_ids:           # HAM ⊂ ISIC2019 -> quitar (ya lo vio el fine-tune)
            ext_dropped_ham += 1
            continue
        ext_items.append((ext_paths[r.id_stem], cls2idx[r.ham]))
    print(f"[Val externa] imagenes usables (no vistas): {len(ext_items)} | "
          f"descartadas por estar en HAM: {ext_dropped_ham}")
    if ext_items:
        from collections import Counter
        dist = Counter(classes[y] for _, y in ext_items)
        print("[Val externa] distribucion mapeada a 7 clases:", dict(dist))
else:
    print("[Val externa] sin ground truth -> se omitira mas adelante (non-fatal).")

# ---- SMOKE: achica el problema para validar el pipeline rapido ----
if SMOKE:
    def _sub(ids, per_class, seed=42):
        yy = np.array([id2label[i] for i in ids]); rng = np.random.RandomState(seed); keep = []
        for c in np.unique(yy):
            ci = np.where(yy == c)[0]
            keep.extend(rng.choice(ci, min(per_class, len(ci)), replace=False))
        return [ids[i] for i in sorted(keep)]
    tr_ids = _sub(tr_ids, 120)
    va_ids = _sub(va_ids, 40)
    ext_items = ext_items[:EXT_VAL_MAXN]
    print(f"[SMOKE] subsample -> train {len(tr_ids)} | val {len(va_ids)} | ext {len(ext_items)}")

# Pesos = raiz del inverso de frecuencia (suaviza; evita sobre-predecir clases raras)
counts = np.bincount([id2label[i] for i in tr_ids], minlength=len(classes))
w = torch.tensor((counts.sum() / np.maximum(counts, 1)) ** 0.5, dtype=torch.float32)
w = (w / w.mean()).to(device)
print("Pesos por clase:", {c: round(float(x), 2) for c, x in zip(classes, w.cpu())})


## 3. SSL domain-adaptive — SimSiam sobre ResNet50 (la contribución)

SimSiam (Chen & He, CVPR 2021): negative-free, **robusto a batch chico y 1 GPU** → ideal para cómputo de estudiante. Se parte de ImageNet (domain-adaptive, no desde cero: la Práctica 1 probó que desde cero pierde). Diagnóstico de colapso por `out_std` en cada época.

In [ ]:
ssl_aug = transforms.Compose([
    transforms.RandomResizedCrop(SSL_RES, scale=(0.2, 1.0)),
    transforms.RandomApply([transforms.ColorJitter(0.4, 0.4, 0.4, 0.1)], p=0.8),
    transforms.RandomGrayscale(p=0.2),
    transforms.RandomApply([transforms.GaussianBlur(9, (0.1, 2.0))], p=0.5),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

class TwoViewDataset(Dataset):
    """Devuelve dos vistas aumentadas de la misma imagen (sin etiqueta)."""
    def __init__(self, paths, tf):
        self.paths = list(paths); self.tf = tf
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, idx):
        img = load_rgb(self.paths[idx])
        return self.tf(img), self.tf(img)

class SimSiam(nn.Module):
    def __init__(self, dim=2048, pred_dim=512):
        super().__init__()
        bb = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)  # backbone ImageNet
        self.feat_dim = bb.fc.in_features  # 2048
        bb.fc = nn.Identity()
        self.backbone = bb
        self.projector = nn.Sequential(
            nn.Linear(self.feat_dim, dim, bias=False), nn.BatchNorm1d(dim), nn.ReLU(inplace=True),
            nn.Linear(dim, dim, bias=False),           nn.BatchNorm1d(dim), nn.ReLU(inplace=True),
            nn.Linear(dim, dim, bias=False),           nn.BatchNorm1d(dim, affine=False))
        self.predictor = nn.Sequential(
            nn.Linear(dim, pred_dim, bias=False), nn.BatchNorm1d(pred_dim), nn.ReLU(inplace=True),
            nn.Linear(pred_dim, dim))

    def forward(self, x1, x2):
        z1 = self.projector(self.backbone(x1))  # -> (B, dim)
        z2 = self.projector(self.backbone(x2))
        p1 = self.predictor(z1)
        p2 = self.predictor(z2)
        return p1, p2, z1.detach(), z2.detach()  # stop-gradient en z

def neg_cos(p, z):
    return -F.cosine_similarity(p, z, dim=1).mean()


In [ ]:
DA_BACKBONE = str(WORK / "backbone_domain_adaptive_r50.pth")

# RESUME: si el backbone SSL ya existe (working de esta sesion, o subido como Input dataset),
# reusarlo y SALTAR el pretrain SSL (~6h). Clave para retomar una corrida que murio tras el SSL.
_bk = None
if Path(DA_BACKBONE).exists():
    _bk = DA_BACKBONE
else:
    import shutil, zipfile
    def _loadable(p):
        # un .pth de torch ES un zip internamente; torch.load lo lee sin importar la extension (.zip incluida)
        try:
            torch.load(p, map_location="cpu"); return True
        except Exception:
            return False
    # 1) .pth o .zip que ya es un archivo torch cargable
    cands = (list(INPUT.glob("**/backbone_domain_adaptive_r50.pth")) +
             list(INPUT.glob("**/backbone_domain_adaptive_r50.zip")) +
             list(INPUT.glob("**/*.pth")))
    for c in cands:
        if _loadable(c):
            shutil.copy(c, DA_BACKBONE); _bk = str(c); break
    # 2) fallback: Kaggle extrajo el zip -> data.pkl + data/ sueltos -> re-armar el archivo torch
    if _bk is None:
        for pkl in INPUT.glob("**/data.pkl"):
            root = pkl.parent
            if list(root.glob("data/*")):
                tmp = str(WORK / "_bk_rezip.pth")
                with zipfile.ZipFile(tmp, "w", zipfile.ZIP_STORED) as zf:
                    for f in root.rglob("*"):
                        if f.is_file():
                            zf.write(f, arcname=f"{root.name}/{f.relative_to(root)}")
                if _loadable(tmp):
                    shutil.move(tmp, DA_BACKBONE); _bk = f"re-zip:{root.name}"; break

if RUN_SSL and _bk and not SMOKE:
    print(f"[SSL] backbone encontrado -> se REUSA (resume, se salta el pretrain de ~6h): {_bk}")
elif RUN_SSL:
    set_seed(42)
    # Pool SSL: ISIC (externo, sin labels) + solo lesiones de TRAIN de HAM (val y ext quedan fuera)
    ssl_paths = [p for p in ssl_extra.values()
                 if p.stem not in external_val_ids and p.stem not in set(va_ids)]
    ssl_paths += [img_paths[i] for i in tr_ids]
    if SMOKE:
        random.Random(42).shuffle(ssl_paths)
        ssl_paths = ssl_paths[:500]   # SMOKE: pool chico -> valida el SSL en segundos, no en minutos
        print(f"[SSL][SMOKE] pool recortado a {len(ssl_paths)} imgs (solo validar que el SSL corre)")
    print(f"[SSL] imagenes en el pool (sin etiquetas): {len(ssl_paths)} "
          f"(ISIC externo + HAM train)")
    ssl_dl = DataLoader(TwoViewDataset(ssl_paths, ssl_aug), batch_size=SSL_BATCH, shuffle=True,
                        num_workers=NUM_WORKERS, pin_memory=True, drop_last=True,
                        persistent_workers=False)  # workers no-persistentes -> liberan RAM entre epochs

    ssl_model = SimSiam().to(device)
    if USE_DP:
        ssl_model = nn.DataParallel(ssl_model)
    base_lr = 0.05 * SSL_BATCH / 256
    ssl_opt = torch.optim.SGD(ssl_model.parameters(), lr=base_lr, momentum=0.9, weight_decay=1e-4)
    ssl_sched = torch.optim.lr_scheduler.CosineAnnealingLR(ssl_opt, T_max=SSL_EPOCHS)
    scaler = torch.amp.GradScaler("cuda")

    def _ssl_backbone():
        return (ssl_model.module if USE_DP else ssl_model).backbone

    t0 = time.time()
    for e in range(SSL_EPOCHS):
        ssl_model.train(); tot = 0.0; last_z = None
        for v1, v2 in ssl_dl:
            v1, v2 = v1.to(device, non_blocking=True), v2.to(device, non_blocking=True)
            ssl_opt.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda"):
                p1, p2, z1, z2 = ssl_model(v1, v2)
                loss = 0.5 * neg_cos(p1, z2) + 0.5 * neg_cos(p2, z1)
            scaler.scale(loss).backward(); scaler.step(ssl_opt); scaler.update()
            tot += loss.item(); last_z = z1
        ssl_sched.step()
        with torch.no_grad():
            zc = F.normalize(last_z.float(), dim=1)
            out_std = zc.std(dim=0).mean().item()   # sano ~1/sqrt(dim)=0.022 ; ~0 -> colapso
        el = time.time() - t0; eta = el / (e + 1) * (SSL_EPOCHS - e - 1)
        print(f"[SSL] Epoch {e+1}/{SSL_EPOCHS} - loss: {tot/len(ssl_dl):.4f} | "
              f"out_std: {out_std:.4f} (colapso si ~0, sano ~0.022) | "
              f"lr: {ssl_sched.get_last_lr()[0]:.2e} | ETA: {eta/60:.1f} min")
        if (e + 1) % 10 == 0:  # checkpoint parcial -> resume si la sesion se cae
            torch.save(_ssl_backbone().state_dict(), DA_BACKBONE)

    torch.save(_ssl_backbone().state_dict(), DA_BACKBONE)
    print(f"[SSL] backbone domain-adaptive guardado -> {DA_BACKBONE} | tiempo SSL: {(time.time()-t0)/60:.1f} min")
    del ssl_model; torch.cuda.empty_cache()
else:
    print("SSL omitido (RUN_SSL=False).")
print("Tiempo total transcurrido:", elapsed())


## 4. Fine-tune supervisado + utilidades de evaluación (KPIs de tesis)

Toda la comparación es justa: mismo split, mismos hiperparámetros, misma semilla por cada corrida de la ablación. Métricas: macro-F1, AUC-ROC, ECE. Evaluación con TTA (hflip).

In [ ]:
train_tf = transforms.Compose([
    transforms.RandomResizedCrop(SUP_RES, scale=(0.6, 1.0)),
    transforms.RandomHorizontalFlip(), transforms.RandomVerticalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(0.2, 0.2, 0.2, 0.05),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
    transforms.RandomErasing(p=0.25),
])
val_tf = transforms.Compose([
    transforms.Resize(256), transforms.CenterCrop(SUP_RES),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

class HAMds(Dataset):
    def __init__(self, ids, tf): self.ids, self.tf = ids, tf
    def __len__(self): return len(self.ids)
    def __getitem__(self, idx):
        i = self.ids[idx]
        return self.tf(load_rgb(img_paths[i])), id2label[i]

train_dl = DataLoader(HAMds(tr_ids, train_tf), batch_size=64, shuffle=True,
                      num_workers=NUM_WORKERS, pin_memory=True, drop_last=True,
                      persistent_workers=False)  # no-persistente -> evita fuga de RAM en runs largos
val_dl = DataLoader(HAMds(va_ids, val_tf), batch_size=128, num_workers=NUM_WORKERS, pin_memory=True)

def build_model(init="imagenet", ssl_path=DA_BACKBONE):
    """init='imagenet' (baseline) o 'ssl' (backbone domain-adaptive)."""
    if init == "ssl":
        net = resnet50(weights=None)
        sd = torch.load(ssl_path, map_location="cpu")
        missing, unexpected = net.load_state_dict(sd, strict=False)  # fc.* falta (esperado)
        assert all(k.startswith("fc.") for k in missing), f"Faltan pesos inesperados: {missing}"
    else:
        net = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
    net.fc = nn.Linear(net.fc.in_features, len(classes))
    return net.to(device)

@torch.no_grad()
def collect_logits(net, dl):
    """Logits promediados con TTA (hflip). Devuelve (N, C) y labels (N,)."""
    net.eval(); LG, GT = [], []
    for imgs, lbls in dl:
        imgs = imgs.to(device)
        with torch.amp.autocast("cuda"):
            logits = (net(imgs) + net(torch.flip(imgs, dims=[3]))) / 2
        LG.append(logits.float().cpu()); GT.append(lbls)
    return torch.cat(LG).numpy(), torch.cat(GT).numpy()

def ece_score(probs, labels, n_bins=15):
    """Expected Calibration Error."""
    conf = probs.max(1); pred = probs.argmax(1)
    acc = (pred == labels).astype(float)
    bins = np.linspace(0, 1, n_bins + 1); ece = 0.0
    for i in range(n_bins):
        m = (conf > bins[i]) & (conf <= bins[i + 1])
        if m.sum() > 0:
            ece += (m.mean()) * abs(acc[m].mean() - conf[m].mean())
    return float(ece)

def metrics_from_logits(logits, labels, temp=1.0):
    probs = softmax(logits / temp, axis=1)
    pred = probs.argmax(1)
    mf1 = f1_score(labels, pred, average="macro")
    acc = accuracy_score(labels, pred)
    try:
        auc = roc_auc_score(labels, probs, multi_class="ovr", average="macro",
                            labels=list(range(len(classes))))
    except Exception:
        auc = float("nan")
    ece = ece_score(probs, labels)
    return dict(macro_f1=mf1, acc=acc, auc=auc, ece=ece), probs, pred


In [ ]:
# ============================================================================
# MEJORA — Fine-tune LP-FT + Layer-wise LR Decay (LLRD)
# Kumar et al. 2022 ("Fine-tuning can distort pretrained features"): el full
# fine-tune INGENUO distorsiona las features pre-entrenadas -> borra la ventaja
# del SSL. LP-FT la preserva:
#   Fase 1 (LP): backbone congelado, se entrena SOLO la cabeza lineal.
#   Fase 2 (FT): se descongela todo, LR chico + layer-wise LR decay.
# Se aplica a AMBOS brazos (comparacion justa). train_arm acepta 'seed' (multi-semilla).
# ============================================================================
LP_EPOCHS  = 2 if SMOKE else 10
FT_EPOCHS  = 2 if SMOKE else 15
LP_LR      = 1e-3
FT_LR      = 1e-4
LLRD_DECAY = 0.75

def _llrd_groups(model, base_lr, decay):
    order = ["fc", "layer4", "layer3", "layer2", "layer1", "bn1", "conv1"]
    groups, seen = [], set()
    for depth, prefix in enumerate(order):
        ps = [p for n, p in model.named_parameters()
              if n.split(".")[0] == prefix and p.requires_grad and id(p) not in seen]
        for p in ps:
            seen.add(id(p))
        if ps:
            groups.append({"params": ps, "lr": base_lr * (decay ** depth)})
    rest = [p for _, p in model.named_parameters()
            if p.requires_grad and id(p) not in seen]
    if rest:
        groups.append({"params": rest, "lr": base_lr * (decay ** len(order))})
    return groups

def _run_phase(net, core, epochs, opt, sched, crit, tag, best):
    scaler = torch.amp.GradScaler("cuda")
    t0 = time.time()
    for e in range(epochs):
        net.train()
        for imgs, lbls in train_dl:
            imgs, lbls = imgs.to(device), lbls.to(device)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda"):
                loss = crit(net(imgs), lbls)
            scaler.scale(loss).backward()
            scaler.step(opt); scaler.update()
        if sched is not None:
            sched.step()
        lg, gt = collect_logits(net, val_dl)
        m, _, _ = metrics_from_logits(lg, gt)
        if m["macro_f1"] > best["f1"]:
            best["f1"] = m["macro_f1"]; best["auc"] = m["auc"]
            best["state"] = {k: v.detach().cpu().clone() for k, v in core.state_dict().items()}
            best["lg"], best["gt"] = lg, gt
        eta = (time.time() - t0) / (e + 1) * (epochs - e - 1)
        print(f"[{tag}] Epoch {e+1}/{epochs} - macro-F1: {m['macro_f1']:.3f} | "
              f"AUC: {m['auc']:.3f} | best: {best['f1']:.3f} | ETA {eta/60:.1f} min")

def train_arm(init, seed=42, tag=""):
    """LP-FT (linear probe -> fine-tune con LLRD). Preserva la representacion SSL.
    Devuelve dict best con f1, auc, state, lg, gt."""
    set_seed(seed)
    net = build_model(init)
    core = net
    if USE_DP:
        net = nn.DataParallel(net)
    crit = nn.CrossEntropyLoss(weight=w, label_smoothing=0.1)
    best = {"f1": 0.0, "auc": float("nan"), "state": None, "lg": None, "gt": None}

    # Fase 1 — LINEAR PROBE
    for p in core.parameters():
        p.requires_grad = False
    for p in core.fc.parameters():
        p.requires_grad = True
    opt = torch.optim.AdamW(core.fc.parameters(), lr=LP_LR, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=LP_EPOCHS)
    _run_phase(net, core, LP_EPOCHS, opt, sched, crit, f"{tag}-LP", best)

    # Fase 2 — FINE-TUNE completo con LLRD
    for p in core.parameters():
        p.requires_grad = True
    opt = torch.optim.AdamW(_llrd_groups(core, FT_LR, LLRD_DECAY), weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=FT_EPOCHS)
    _run_phase(net, core, FT_EPOCHS, opt, sched, crit, f"{tag}-FT", best)

    if best["state"] is None:
        best["state"] = {k: v.detach().cpu().clone() for k, v in core.state_dict().items()}
        best["lg"], best["gt"] = collect_logits(net, val_dl)
    del net, core
    torch.cuda.empty_cache()
    return best

print(f"train_arm -> LP-FT + LLRD (Kumar 2022) | LP:{LP_EPOCHS} FT:{FT_EPOCHS} | LLRD:{LLRD_DECAY}")


## 5. Ablación multi-semilla — Baseline ImageNet vs SSL domain-adaptive

Cada brazo se entrena con **N_SEEDS semillas**. Se reporta **media ± std** de macro-F1 y AUC → el resultado ya no depende del ruido de una sola corrida. El **modelo entregable** es el mejor brazo SSL. Export incremental: apenas hay un modelo, se guarda a disco.

In [ ]:
def DEPLOY_INIT():
    return 'ssl' if RUN_SSL else 'imagenet'

results = {}        # nombre -> (best_f1, best_state, lg, gt)  [compat con celdas siguientes]
seed_stats = {}     # nombre -> {"f1": [...], "auc": [...]}
# El brazo ENTREGABLE (ssl) va PRIMERO -> su modelo se exporta temprano; si algo muere despues,
# best_resnet50_ssl.pth ya quedo escrito en /kaggle/working (Kaggle commitea outputs parciales).
arms = ["ssl", "imagenet"] if RUN_SSL else ["imagenet"]

for name in arms:
    seed_stats[name] = {"f1": [], "auc": []}
    best_overall = {"f1": -1.0}
    for s in SEEDS:
        if (time.time() - T_START) / 3600 > MAX_HOURS and s != SEEDS[0]:
            print(f"[GUARDA TIEMPO] {elapsed()} > {MAX_HOURS}h -> se SALTAN semillas extra de '{name}'. "
                  f"Reportado con {len(seed_stats[name]['f1'])} semilla(s). (no es un corte silencioso)")
            break
        tag = f"{name.upper()}-s{s}"
        print(f"\n===== {tag} =====")
        b = train_arm(name, seed=s, tag=tag)
        seed_stats[name]["f1"].append(float(b["f1"]))
        seed_stats[name]["auc"].append(float(b["auc"]))
        if b["f1"] > best_overall["f1"]:
            best_overall = b
        # export incremental: guardar SIEMPRE el mejor SSL a disco apenas exista
        if name == DEPLOY_INIT():
            torch.save({"model_state_dict": best_overall["state"], "classes": classes,
                        "arch": "resnet50", "img_size": SUP_RES, "init": name},
                       str(WORK / "best_resnet50_ssl.pth"))
    results[name] = (best_overall["f1"], best_overall["state"], best_overall["lg"], best_overall["gt"])
    gc.collect(); torch.cuda.empty_cache()   # libera RAM/VRAM entre brazos (evita OOM en runs largos)

def _ms(xs):
    xs = [x for x in xs if x == x]  # drop NaN
    return (float(np.mean(xs)), float(np.std(xs))) if xs else (float("nan"), float("nan"))

print("\n\n================ ABLACION MULTI-SEMILLA (media +/- std) ================")
print(f"{'Backbone':<22}{'macro-F1':>18}{'AUC':>18}   (n semillas)")
ablation_ms = {}
for name in arms:
    f1m, f1s = _ms(seed_stats[name]["f1"]); aucm, aucs = _ms(seed_stats[name]["auc"])
    ablation_ms[name] = {"f1_mean": f1m, "f1_std": f1s, "auc_mean": aucm, "auc_std": aucs,
                         "n_seeds": len(seed_stats[name]["f1"]), "seeds": SEEDS[:len(seed_stats[name]["f1"])]}
    label = "ImageNet (baseline)" if name == "imagenet" else "SSL domain-adaptive"
    print(f"{label:<22}{f1m:>8.4f} +/-{f1s:<6.4f}{aucm:>8.4f} +/-{aucs:<6.4f}   (n={ablation_ms[name]['n_seeds']})")
if RUN_SSL:
    df1 = ablation_ms["ssl"]["f1_mean"] - ablation_ms["imagenet"]["f1_mean"]
    dauc = ablation_ms["ssl"]["auc_mean"] - ablation_ms["imagenet"]["auc_mean"]
    print(f"\nDelta media (SSL - ImageNet): macro-F1 {df1:+.4f} | AUC {dauc:+.4f}")

# El MODELO ENTREGABLE del proyecto = SSL domain-adaptive (nucleo de la tesis).
DEPLOY = "ssl" if RUN_SSL else "imagenet"
best_name = DEPLOY
best_f1, best_state, val_logits, val_labels = results[best_name]
print(f"\nMODELO ENTREGABLE (se evalua, calibra y exporta) -> {best_name} (macro-F1 {best_f1:.4f})")
save_json({"ablation_multiseed": ablation_ms, "deploy": best_name}, str(WORK / "RESULTS_SUMMARY.json"))
print("Checkpoint incremental + resumen parcial guardados. Tiempo:", elapsed())


## 6. Reporte clínico + calibración (ECE) + screening de melanoma

KPIs de tesis sobre el **modelo entregable (SSL)**.

In [ ]:
m0, probs0, pred0 = metrics_from_logits(val_logits, val_labels)
print(f"Backbone entregable: {best_name} | macro-F1: {m0['macro_f1']:.4f} | acc: {m0['acc']:.4f} | "
      f"AUC: {m0['auc']:.4f} | ECE(sin calibrar): {m0['ece']:.4f}\n")
print(classification_report(val_labels, pred0, target_names=classes, digits=3, zero_division=0))
print("Matriz de confusion (filas=real, cols=pred):")
print(pd.DataFrame(confusion_matrix(val_labels, pred0), index=classes, columns=classes))

def fit_temperature(logits, labels):
    # T acotada a [0.5, 5.0] -> evita que un val chico/ruidoso reviente la calibracion (T->0)
    lg = torch.tensor(logits); lb = torch.tensor(labels)
    T = torch.nn.Parameter(torch.ones(1))
    opt = torch.optim.LBFGS([T], lr=0.05, max_iter=100)
    def closure():
        opt.zero_grad(); loss = F.cross_entropy(lg / T.clamp(0.5, 5.0), lb); loss.backward(); return loss
    opt.step(closure)
    return float(T.detach().clamp(0.5, 5.0))

T_fit = fit_temperature(val_logits, val_labels)
ece_try = metrics_from_logits(val_logits, val_labels, temp=T_fit)[0]["ece"]
# Guarda: solo se calibra si REALMENTE mejora el ECE; si no, T=1 (no calibrar)
if ece_try <= m0["ece"]:
    T = T_fit
else:
    print(f"[Calibracion] T={T_fit:.3f} empeoraba el ECE ({m0['ece']:.4f}->{ece_try:.4f}) -> se usa T=1.0")
    T = 1.0
m_cal, probs_cal, _ = metrics_from_logits(val_logits, val_labels, temp=T)
print(f"\nTemperatura usada T = {T:.3f}")
print(f"ECE  antes: {m0['ece']:.4f}  ->  despues: {m_cal['ece']:.4f}")

mi = classes.index("mel")
y_mel = (val_labels == mi).astype(int)
fpr, tpr, thr = roc_curve(y_mel, probs_cal[:, mi])
ok = np.where(tpr >= 0.80)[0]
mel_thr = float(thr[ok[0]]) if len(ok) else None
if len(ok):
    j = ok[0]
    print(f"\n[Screening mel] umbral prob>={thr[j]:.3f} -> sensibilidad={tpr[j]:.3f} | "
          f"especificidad={1-fpr[j]:.3f}")
else:
    print("\n[Screening mel] no se alcanza sens>=0.80 en la curva ROC (revisar).")
sens_argmax = (pred0[val_labels == mi] == mi).mean()
print(f"[Argmax mel] sensibilidad con decision estandar: {sens_argmax:.3f}")

print("\n================= KPIs DE TESIS =================")
def kpi(name, val, thr_, ge=True):
    okk = val >= thr_ if ge else val <= thr_
    print(f"  {name:<34}{val:>7.3f}  (meta {'>=' if ge else '<='}{thr_})  {'PASA' if okk else 'NO PASA'}")
kpi("macro-F1", m0["macro_f1"], 0.65)
kpi("AUC-ROC macro", m0["auc"], 0.85)
kpi("Sensibilidad melanoma (screening)", tpr[ok[0]] if len(ok) else sens_argmax, 0.80)
kpi("ECE (calibrado)", m_cal["ece"], 0.10, ge=False)


## 6b. Prueba REAL de la contribución SSL — Linear probe + pocas etiquetas

El full fine-tune tapa la calidad de la representación (re-entrena todo). El linear probe con backbone **congelado** la aísla. Non-fatal: si falla, la corrida sigue.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

probe = {}
try:
    def build_backbone(init):
        if init == "ssl":
            net = resnet50(weights=None)
            net.load_state_dict(torch.load(DA_BACKBONE, map_location="cpu"), strict=False)
        else:
            net = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
        net.fc = nn.Identity()
        return net.to(device).eval()

    @torch.no_grad()
    def extract_features(backbone, ids):
        dl = DataLoader(HAMds(ids, val_tf), batch_size=128, num_workers=NUM_WORKERS, pin_memory=True)
        FE, YE = [], []
        for imgs, lbls in dl:
            imgs = imgs.to(device)
            with torch.amp.autocast("cuda"):
                f = backbone(imgs)
            FE.append(f.float().cpu()); YE.append(lbls)
        return torch.cat(FE).numpy(), torch.cat(YE).numpy()

    def strat_subsample(yy, frac, seed=42):
        rng = np.random.RandomState(seed); idx = []
        for c in np.unique(yy):
            ci = np.where(yy == c)[0]
            n = max(1, int(round(len(ci) * frac)))
            idx.extend(rng.choice(ci, n, replace=False))
        return np.array(idx)

    FRACS = [0.1, 0.2, 0.5, 1.0]
    parms = ["imagenet", "ssl"] if RUN_SSL else ["imagenet"]
    probe = {a: {} for a in parms}
    for init in parms:
        bb = build_backbone(init)
        Xtr, ytr = extract_features(bb, tr_ids)
        Xva, yva = extract_features(bb, va_ids)
        del bb; torch.cuda.empty_cache()
        sc = StandardScaler().fit(Xtr)
        Xtr_s, Xva_s = sc.transform(Xtr), sc.transform(Xva)
        for frac in FRACS:
            sub = strat_subsample(ytr, frac)
            clf = LogisticRegression(max_iter=2000, class_weight="balanced", C=1.0)
            clf.fit(Xtr_s[sub], ytr[sub])
            prob = clf.predict_proba(Xva_s); pred = prob.argmax(1)
            mf1 = f1_score(yva, pred, average="macro")
            auc = roc_auc_score(yva, prob, multi_class="ovr", average="macro",
                                labels=list(range(len(classes))))
            probe[init][frac] = (float(mf1), float(auc))
            print(f"[LP {init:>8}] labels={int(frac*100):>3}% -> macro-F1: {mf1:.3f} | "
                  f"AUC: {auc:.3f} | n_train={len(sub)}")

    if RUN_SSL:
        print("\n===== LINEAR PROBE: SSL vs ImageNet (backbone congelado) =====")
        print(f"{'% labels':>9}{'IN F1':>9}{'SSL F1':>9}{'dF1':>8}{'IN AUC':>9}{'SSL AUC':>9}")
        for frac in FRACS:
            inf, ina = probe["imagenet"][frac]; ssf, ssa = probe["ssl"][frac]
            print(f"{int(frac*100):>8}%{inf:>9.3f}{ssf:>9.3f}{ssf-inf:>+8.3f}{ina:>9.3f}{ssa:>9.3f}")
        print("\ndF1/dAUC > 0 => el SSL aporta representacion superior a ImageNet en ese regimen.")
except Exception as e:
    print("[Linear probe] error (non-fatal):", repr(e))
print("Tiempo:", elapsed())


## 6c. Fiabilidad — calibración visual, intervalos de confianza, abstención

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# Reconstruir el modelo entregable (para eval externa y Grad-CAM)
eval_net = resnet50(weights=None)
eval_net.fc = nn.Linear(eval_net.fc.in_features, len(classes))
eval_net.load_state_dict(best_state); eval_net.to(device).eval()

def reliability_diagram(probs, labels, n_bins=12, title="Calibracion", fname=None):
    conf = probs.max(1); pred = probs.argmax(1); acc = (pred == labels).astype(float)
    bins = np.linspace(0, 1, n_bins + 1); xs, ys = [], []
    for i in range(n_bins):
        m = (conf > bins[i]) & (conf <= bins[i + 1])
        if m.sum() > 0:
            xs.append(conf[m].mean()); ys.append(acc[m].mean())
    plt.figure(figsize=(4, 4))
    plt.plot([0, 1], [0, 1], "--", color="gray", label="perfecto")
    plt.plot(xs, ys, "o-", label="modelo")
    plt.xlabel("Confianza media"); plt.ylabel("Exactitud real"); plt.title(title)
    plt.legend(); plt.grid(alpha=0.3)
    if fname: plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.close()

reliability_diagram(probs_cal, val_labels, title=f"Calibracion ({best_name}, T={T:.2f})",
                    fname=str(WORK / "reliability_diagram.png"))

def bootstrap_ci(logits, labels, metric_fn, n=1000, seed=42):
    rng = np.random.RandomState(seed); N = len(labels); vals = []
    for _ in range(n):
        idx = rng.randint(0, N, N)
        vals.append(metric_fn(logits[idx], labels[idx]))
    vals = [v for v in vals if v == v]
    return float(np.mean(vals)), float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))

f1_fn = lambda lg, lb: f1_score(lb, softmax(lg, axis=1).argmax(1), average="macro")
mel_fn = lambda lg, lb: ((softmax(lg, axis=1).argmax(1) == mi)[lb == mi]).mean() if (lb == mi).any() else np.nan
mu_f1, lo_f1, hi_f1 = bootstrap_ci(val_logits, val_labels, f1_fn)
mu_ms, lo_ms, hi_ms = bootstrap_ci(val_logits, val_labels, mel_fn)
print(f"macro-F1         : {mu_f1:.3f}  IC95% [{lo_f1:.3f}, {hi_f1:.3f}]")
print(f"sens mel (argmax): {mu_ms:.3f}  IC95% [{lo_ms:.3f}, {hi_ms:.3f}]")

print("\n== Abstencion: responde solo si confia; el resto -> especialista ==")
conf = probs_cal.max(1); pred_c = probs_cal.argmax(1); abst = {}
for tau in [0.0, 0.5, 0.6, 0.7, 0.8, 0.9]:
    keep = conf >= tau
    cov = float(keep.mean())
    acc = float((pred_c[keep] == val_labels[keep]).mean()) if keep.sum() else float("nan")
    abst[tau] = (cov, acc)
    print(f"  umbral>={tau:.1f} -> cobertura {cov*100:5.1f}% | exactitud en lo respondido {acc*100:5.1f}%")


## 6d. Validación externa cross-dataset (ISIC 2019, anti-leakage)

La prueba de generalización más fuerte: el modelo se evalúa en **imágenes de otro dataset que NUNCA vio** (ni en SSL ni en fine-tune; se excluyeron las que están en HAM). Non-fatal.

In [ ]:
ext_result = None
try:
    if ext_items:
        n_use = len(ext_items) if EXT_VAL_MAXN is None else min(EXT_VAL_MAXN, len(ext_items))
        # subsample estratificado para respetar el cap sin sesgar clases
        if n_use < len(ext_items):
            yy = np.array([y for _, y in ext_items]); rng = np.random.RandomState(42); keep = []
            per = max(1, n_use // len(classes))
            for c in range(len(classes)):
                ci = np.where(yy == c)[0]
                keep.extend(rng.choice(ci, min(per, len(ci)), replace=False))
            keep = keep[:n_use]
            use_items = [ext_items[i] for i in keep]
        else:
            use_items = ext_items

        class ExtDS(Dataset):
            def __init__(self, items): self.items = items
            def __len__(self): return len(self.items)
            def __getitem__(self, i):
                p, yy = self.items[i]
                return val_tf(load_rgb(p)), yy
        edl = DataLoader(ExtDS(use_items), batch_size=128, num_workers=NUM_WORKERS, pin_memory=True)
        elg, egt = collect_logits(eval_net, edl)
        em, eprobs, epred = metrics_from_logits(elg, egt, temp=T)

        erec = recall_score(egt, epred, average=None, labels=list(range(len(classes))), zero_division=0)
        # screening de melanoma en dominio externo, con el MISMO umbral calibrado en val
        ext_mel_sens = ext_mel_spec = None
        if mel_thr is not None:
            pm = eprobs[:, mi]; ymel = (egt == mi).astype(int)
            if ymel.sum() > 0:
                ext_mel_sens = float(((pm >= mel_thr)[ymel == 1]).mean())
                ext_mel_spec = float(((pm < mel_thr)[ymel == 0]).mean())

        ext_result = {"n": int(len(use_items)), "macro_f1": float(em["macro_f1"]),
                      "auc": float(em["auc"]), "acc": float(em["acc"]), "ece": float(em["ece"]),
                      "recall_por_clase": {classes[i]: round(float(erec[i]), 4) for i in range(len(classes))},
                      "mel_screening_sens": ext_mel_sens, "mel_screening_spec": ext_mel_spec}
        print(f"[Val externa · ISIC2019 no visto] n={ext_result['n']}")
        print(f"  macro-F1: {em['macro_f1']:.4f} | AUC: {em['auc']:.4f} | acc: {em['acc']:.4f} | ECE: {em['ece']:.4f}")
        print("  recall por clase:", ext_result["recall_por_clase"])
        if ext_mel_sens is not None:
            print(f"  screening melanoma (umbral val {mel_thr:.3f}): sens={ext_mel_sens:.3f} | spec={ext_mel_spec:.3f}")
        drop = em["macro_f1"] - m0["macro_f1"]
        print(f"  Caida vs val interna (macro-F1): {drop:+.4f}  "
              f"({'generaliza bien' if drop > -0.10 else 'caida notable -> discutir domain shift'})")
    else:
        print("[Val externa] OMITIDA -> no hay dataset externo etiquetado (adjunta ISIC2019 con su GroundTruth).")
except Exception as e:
    print("[Val externa] error (non-fatal):", repr(e))
print("Tiempo:", elapsed())


## 6e. Grad-CAM — ¿el modelo mira la lesión o un artefacto?

In [ ]:
try:
    class GradCAM:
        def __init__(self, model, layer):
            self.model = model; self.a = None; self.g = None
            layer.register_forward_hook(lambda m, i, o: setattr(self, "a", o.detach()))
            layer.register_full_backward_hook(lambda m, gi, go: setattr(self, "g", go[0].detach()))
        def __call__(self, x, cls):
            self.model.zero_grad()
            out = self.model(x)
            out[0, cls].backward()
            wts = self.g.mean(dim=(2, 3), keepdim=True)
            cam = (wts * self.a).sum(1, keepdim=True).clamp(min=0)
            cam = F.interpolate(cam, size=x.shape[2:], mode="bilinear", align_corners=False)[0, 0]
            cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
            return cam.cpu().numpy(), float(out.softmax(1)[0, cls])

    def denorm(x):
        t = x.detach().cpu().numpy().transpose(1, 2, 0)
        return np.clip(t * np.array(STD) + np.array(MEAN), 0, 1)

    cam = GradCAM(eval_net, eval_net.layer4[-1])
    mel_val_ids = [i for i in va_ids if id2label[i] == mi][:3]
    if not mel_val_ids:
        print("[Grad-CAM] no hay melanomas en val (posible en SMOKE) -> se omite.")
    else:
        fig, axes = plt.subplots(len(mel_val_ids), 2, figsize=(6, 3 * len(mel_val_ids)))
        if len(mel_val_ids) == 1:
            axes = axes[None, :]
        for r, iid in enumerate(mel_val_ids):
            x = val_tf(load_rgb(img_paths[iid])).unsqueeze(0).to(device)
            heat, prob = cam(x, mi)
            img = denorm(x[0])
            axes[r, 0].imshow(img); axes[r, 0].set_title(f"{iid} (mel real)"); axes[r, 0].axis("off")
            axes[r, 1].imshow(img); axes[r, 1].imshow(heat, cmap="jet", alpha=0.45)
            axes[r, 1].set_title(f"Grad-CAM  p(mel)={prob:.2f}"); axes[r, 1].axis("off")
        plt.tight_layout()
        plt.savefig(str(WORK / "gradcam_melanomas.png"), dpi=150, bbox_inches="tight"); plt.close()
        print("Grad-CAM guardado. El calor debe caer SOBRE la lesion, no en bordes / reglas / pelos.")
except Exception as e:
    print("[Grad-CAM] error (non-fatal):", repr(e))


## 7. Exportar el modelo entregable + prueba de inferencia (para la API)

In [ ]:
OUT = str(WORK / "best_resnet50_ssl.pth")
torch.save({
    "model_state_dict": best_state,
    "classes": classes,
    "arch": "resnet50",
    "img_size": SUP_RES,
    "init": best_name,
    "macro_f1": float(best_f1),
    "temperature": float(T),
    "mel_screening_threshold": mel_thr,
    "norm_mean": MEAN, "norm_std": STD,
}, OUT)
print("Modelo entregable guardado en", OUT)

ckpt = torch.load(OUT, map_location=device, weights_only=False)
best_net = resnet50(weights=None)
best_net.fc = nn.Linear(best_net.fc.in_features, len(ckpt["classes"]))
best_net.load_state_dict(ckpt["model_state_dict"]); best_net.to(device).eval()

@torch.no_grad()
def _infer(img, temp):
    x = val_tf(img).unsqueeze(0).to(device)
    logits = (best_net(x) + best_net(torch.flip(x, dims=[3]))) / 2   # TTA hflip
    p = F.softmax(logits / temp, dim=1)[0].cpu()
    i = int(p.argmax())
    out = {"pred": ckpt["classes"][i], "conf": float(p[i]),
           "probs": {c: round(float(p[j]), 4) for j, c in enumerate(ckpt["classes"])}}
    thr_mel = ckpt.get("mel_screening_threshold")
    if thr_mel is not None and "mel" in ckpt["classes"]:
        pm = float(p[ckpt["classes"].index("mel")])
        out["mel_prob"] = round(pm, 4)
        out["mel_alert"] = bool(pm >= thr_mel)
    return out

def predict(image_id, temp=None):
    return _infer(load_rgb(img_paths[image_id]), ckpt["temperature"] if temp is None else temp)
def predict_path(path, temp=None):
    return _infer(load_rgb(path), ckpt["temperature"] if temp is None else temp)

mel_id = meta[meta.dx == "mel"].image_id.iloc[0]
print(f"\nPrueba inferencia -> Real: mel | {predict(mel_id)}")
print("\nUso en la API:  predict_path('/ruta/a/imagen.jpg')")


## 8. RESUMEN CONSOLIDADO — pegá esto en el chat para revisión

In [ ]:
rec = recall_score(val_labels, pred0, average=None, labels=list(range(len(classes))), zero_division=0)

summary = {
    "modo": "SMOKE (numeros NO validos)" if SMOKE else "corrida real",
    "tiempo_total": elapsed(),
    "dataset": {
        "train_imgs": len(tr_ids), "val_imgs": len(va_ids),
        "classes": classes, "lesion_overlap_train_val": int(len(overlap)),
        "split": "StratifiedGroupKFold por lesion_id (anti-leakage)",
        "ssl_pool_extra_imgs": len(ssl_extra),
        "val_externa_usable": len(ext_items),
    },
    "ssl": {"metodo": "SimSiam domain-adaptive", "backbone": "resnet50", "epochs": SSL_EPOCHS,
            "res": SSL_RES, "batch": SSL_BATCH, "run": bool(RUN_SSL)},
    "ablacion_multiseed": ablation_ms,
    "modelo_entregable": best_name,
    "final_metrics_val_interna": {
        "macro_f1": round(float(m0["macro_f1"]), 4), "acc": round(float(m0["acc"]), 4),
        "auc": round(float(m0["auc"]), 4),
        "ece_raw": round(float(m0["ece"]), 4), "ece_calibrated": round(float(m_cal["ece"]), 4),
        "temperature": round(float(T), 4),
        "macro_f1_ci95": [round(lo_f1, 4), round(hi_f1, 4)],
        "mel_sens_argmax_ci95": [round(lo_ms, 4), round(hi_ms, 4)],
    },
    "recall_por_clase_val": {classes[i]: round(float(rec[i]), 4) for i in range(len(classes))},
    "mel_screening": {"threshold": mel_thr,
                      "sensibilidad": round(float(tpr[ok[0]]), 4) if len(ok) else None,
                      "especificidad": round(float(1 - fpr[ok[0]]), 4) if len(ok) else None},
    "abstencion": {f"{k:.1f}": {"cobertura": round(v[0], 4), "exactitud": round(v[1], 4)}
                   for k, v in abst.items()},
    "linear_probe": ({a: {f"{int(f*100)}pct": {"macro_f1": round(probe[a][f][0], 4),
                                                "auc": round(probe[a][f][1], 4)} for f in probe[a]}
                      for a in probe} if probe else "no ejecutado"),
    "validacion_externa": ext_result if ext_result else "no ejecutada (falta dataset externo etiquetado)",
    "kpis": {
        "macro_f1>=0.65": bool(m0["macro_f1"] >= 0.65),
        "auc>=0.85": bool(m0["auc"] >= 0.85),
        "mel_sens>=0.80": bool((tpr[ok[0]] if len(ok) else 0) >= 0.80),
        "ece_calibrado<=0.10": bool(m_cal["ece"] <= 0.10),
    },
    "artefactos": ["best_resnet50_ssl.pth", "backbone_domain_adaptive_r50.pth",
                   "reliability_diagram.png", "gradcam_melanomas.png", "RESULTS_SUMMARY.json"],
}
save_json(summary, str(WORK / "RESULTS_SUMMARY.json"))
print("=" * 60)
print("RESUMEN CONSOLIDADO — pegalo en el chat para revision")
print("=" * 60)
print(json.dumps(summary, indent=2, ensure_ascii=False, default=float))
